# Build dataset Historical Fact Verification


`input_chunks.json` → Gemini sinh claim → `draft_dataset_for_human_review.json`


In [ ]:
!pip install -q -U google-genai --upgrade

## config

In [ ]:
import json
import os
import re
import time
from collections import Counter
from pathlib import Path

from google import genai
from google.genai import types as genai_types
from google.colab import userdata

INPUT_PATH = "input_chunks.json"
OUTPUT_DIR = Path("dataset_output")

GEMINI_RAW_OUTPUTS_PATH = OUTPUT_DIR / "gemini_raw_outputs.json"
DRAFT_DATASET_PATH = OUTPUT_DIR / "draft_dataset_for_human_review.json"
PASS1_FORMAT_ISSUES_PATH = OUTPUT_DIR / "pass1_format_issues.json"
DATASET_STATS_PATH = OUTPUT_DIR / "dataset_stats.json"

GEMINI_MODEL = "gemini-3.1-flash-lite"
GEMINI_API_KEY_SECRET_NAME = "GEMINI_API_KEY"

MAX_CHUNKS_FOR_TEST = None
REQUEST_SLEEP_SECONDS = 1.0
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 2.0
GEMINI_TEMPERATURE = 0.4

VALID_LABELS = {"SUPPORTED", "REFUTED", "NOT_ENOUGH_EVIDENCE"}
VALID_CLAIM_TYPES = {
    "TIME_EVENT",
    "PERSON_ROLE",
    "GENEALOGY_SUCCESSION",
    "POLITICAL_ADMINISTRATIVE",
    "MILITARY_DIPLOMACY",
    "NATURAL_OMEN_DISASTER",
}
LABEL_ORDER = ["SUPPORTED", "REFUTED", "NOT_ENOUGH_EVIDENCE"]
REQUIRED_CHUNK_FIELDS = ["chunk_id", "pages", "raw_text"]

print("Đã nạp cấu hình.")

def load_gemini_api_key(secret_name=GEMINI_API_KEY_SECRET_NAME):
    try:
        api_key = userdata.get(secret_name)
        if api_key:
            return api_key
    except Exception:
        pass

    api_key = os.environ.get(secret_name)
    if api_key:
        return api_key

    raise RuntimeError(f"Không tìm thấy API key '{secret_name}'.")

GEMINI_API_KEY = load_gemini_api_key()
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

def log(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}")

def ensure_output_dir():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path, data):
    ensure_output_dir()
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def extract_json_from_text(text):
    if text is None:
        return None

    cleaned = text.strip()
    fence = re.search(r"```(?:json)?\s*(.*?)\s*```", cleaned, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        cleaned = fence.group(1).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(cleaned[start:end + 1])
        except json.JSONDecodeError:
            return None

    return None

def normalize_label(value):
    if not isinstance(value, str):
        return None
    value = value.strip().upper().replace(" ", "_").replace("-", "_")
    return value if value in VALID_LABELS else None

def normalize_claim_type(value):
    if not isinstance(value, str):
        return None
    value = value.strip().upper().replace(" ", "_").replace("-", "_")
    return value if value in VALID_CLAIM_TYPES else None

Đã nạp cấu hình.


## prompt

In [ ]:
GEMINI_GENERATION_SYSTEM_PROMPT = """Bạn là chuyên gia tạo dữ liệu cho bài toán Historical Fact Verification (kiểm chứng thông tin lịch sử) dựa trên sử liệu tiếng Việt trích từ Đại Việt Sử Ký Toàn Thư.

NHIỆM VỤ: Từ một đoạn nguồn sử liệu (source context), hãy tạo tối đa 3 claim, mỗi claim ứng với một trong ba nhãn: SUPPORTED, REFUTED, NOT_ENOUGH_EVIDENCE. Tối đa 1 claim cho mỗi nhãn. Không bắt buộc phải tạo đủ cả 3. Nếu đoạn nguồn không đủ thông tin để tạo claim đạt chuẩn cho một nhãn nào đó, hãy bỏ qua nhãn đó. Nếu toàn bộ đoạn nguồn không phù hợp để tạo claim nào, trả về claims rỗng.

=== Ý NGHĨA 3 NHÃN ===
- SUPPORTED: claim được đoạn nguồn hỗ trợ trực tiếp hoặc đủ rõ để kết luận là đúng.
- REFUTED: claim mâu thuẫn với đoạn nguồn (sai nhân vật, sai thời gian, sai địa điểm, sai chức vụ, sai quan hệ, sai kết quả sự kiện, sai nội dung chính sách...).
- NOT_ENOUGH_EVIDENCE: claim có liên quan đến nhân vật/sự kiện/bối cảnh trong đoạn nguồn, nhưng đoạn nguồn không đủ thông tin để kết luận đúng hay sai. Không tạo NEI quá xa ngữ cảnh(quan trọng).

=== NGUYÊN TẮC CHUNG CHO MỌI CLAIM ===
1. Claim là phát biểu khách quan, có thể kiểm chứng.
2. Claim chỉ chứa một ý chính, không gộp nhiều sự kiện độc lập.
3. Claim phải hiểu được độc lập, không phụ thuộc đại từ ("ông ấy", "vua ấy", "người đó",...) hay ngữ cảnh mơ hồ.
4. Claim phải có liên hệ rõ với đoạn nguồn.
5. Claim KHÔNG được sao chép nguyên văn hoặc gần nguyên văn câu trong đoạn nguồn — phải biến đổi cách diễn đạt (đổi cấu trúc câu, đổi trật tự, diễn đạt lại, rút gọn, làm rõ chủ thể nếu nguồn đủ thông tin). Được phép giữ nguyên: tên người, địa danh, niên đại, niên hiệu, chức vụ, tước hiệu, tên triều đại, tên sự kiện.
6. Claim không chứa đánh giá chủ quan hoặc khó kiểm chứng như "vĩ đại nhất", "tốt nhất", "tàn ác", "anh minh", "xấu xa", "nổi tiếng với", "có công lao to lớn", trừ khi claim quy thuộc rõ đây là nhận xét của nguồn hoặc sử thần.
7. Claim không được thêm thông tin ngoài nguồn một cách tùy tiện. Riêng với NOT_ENOUGH_EVIDENCE, chỉ được thêm một chi tiết mở rộng gần với nhân vật/sự kiện/bối cảnh trong nguồn, không tạo giả thuyết xa hoặc sự kiện mới.

=== QUY TẮC RIÊNG CHO TỪNG NHÃN ===

[SUPPORTED]
Lấy một sự kiện/quan hệ/chức vụ/thời gian/địa điểm/kết quả có trong nguồn, viết lại bằng cách diễn đạt khác (paraphrase, rút gọn, làm rõ chủ thể, đổi cấu trúc câu) nhưng giữ đúng nội dung cốt lõi, không thêm thông tin mới, không suy diễn nhân quả nếu nguồn không nói rõ.
Ví dụ tốt — Nguồn: "Huệ Tông không có con trai nối, lập làm hoàng thái tử để truyền ngôi." -> Claim: "Huệ Tông lập Chiêu Hoàng làm hoàng thái tử vì không có con trai nối ngôi."
Ví dụ KHÔNG tốt (quá gần nguồn) — Nguồn nói "nhà Trần đặt chức Hà đê sứ năm 1248" thì không nên viết claim gần như y hệt "Nhà Trần đặt chức Hà đê sứ năm 1248."

[REFUTED]
Tạo claim bằng cách thay đổi MỘT chi tiết kiểm chứng chính so với nguồn, ví dụ nhân vật, thời gian, địa điểm, chức vụ, quan hệ, nguyên nhân, kết quả hoặc đối tượng sự kiện. Các chi tiết còn lại phải giữ gần với nguồn để nguồn có thể bác bỏ claim rõ ràng. Không thay đổi đồng thời quá nhiều chi tiết làm claim trở thành một sự kiện khác.
Không được xem là REFUTED nếu claim chỉ:
- dùng từ đồng nghĩa hoặc diễn đạt lại;
- làm rõ chức năng vốn đã được source nói;
- kế thừa mốc thời gian từ câu/đoạn trước;
- thêm chi tiết mà source không nói nhưng cũng không bác bỏ.
Ví dụ tốt (Sai địa điểm):
Source: "Mùa thu, tháng 8, Hưng Đạo Vương Trần Quốc Tuấn mất ở Vạn Kiếp."
Claim REFUTED: "Vào tháng 8 mùa thu, Hưng Đạo Vương Trần Quốc Tuấn đã qua đời tại kinh thành Thăng Long."
Lý do: Đã biến đổi cách diễn đạt (đổi "mất ở" thành "đã qua đời tại", đảo cấu trúc nhẹ), đồng thời chỉ thay đổi duy nhất fact địa điểm (Vạn Kiếp -> Thăng Long) để nguồn có thể bác bỏ trực tiếp.

[NOT_ENOUGH_EVIDENCE]
Tạo claim có anchor rõ trong source, tức là claim phải bám vào cùng nhân vật, sự kiện, hành động, địa điểm, thời gian hoặc bối cảnh chính đã xuất hiện trong source. Claim NEI không được tách khỏi ngữ cảnh gốc.
Các kiểu tạo NEI phù hợp:

Thêm thông tin nguyên nhân chưa có trong source
Source ghi sự kiện xảy ra, nhưng không nói nguyên nhân.
Thêm động cơ hoặc mục đích chưa có trong source
Source ghi hành động, nhưng không nói mục đích.
Thêm chi tiết định lượng chưa có trong source
Source ghi có quân đội, dân chúng, người dự thi, nhưng không ghi số lượng cụ thể.
Thêm địa điểm phụ chưa có trong source
Source ghi sự kiện nhưng không nói địa điểm chi tiết.
Thêm kết quả lâu dài chưa có trong source
Source ghi sự kiện trước mắt, nhưng không nói hệ quả về sau.
Thêm đánh giá hoặc nhận định không có trong source
Chỉ dùng nếu claim vẫn có thể kiểm chứng bằng corpus rộng hơn, nhưng chunk hiện tại chưa đủ.
Thêm quan hệ chưa được source xác nhận
Source nhắc hai nhân vật, nhưng không nói họ có quan hệ thân tộc, thầy trò, đồng minh hoặc đối địch.

Nếu nguồn đã nêu rõ một thông tin A mà claim thay bằng thông tin B khác, thì claim là REFUTED, không phải NOT_ENOUGH_EVIDENCE.

=== DANH SÁCH CLAIM_TYPE HỢP LỆ (chọn đúng 1 cho mỗi claim) ===
- TIME_EVENT: trọng tâm claim là thời điểm xảy ra sự kiện (năm, tháng, mùa, niên hiệu, đời vua).
- PERSON_ROLE: trọng tâm claim là nhân vật, chức vụ, tước hiệu, vai trò, hành động chính trị/quân sự của một người.
- GENEALOGY_SUCCESSION: trọng tâm claim là quan hệ cha con/mẹ con/vợ chồng/dòng dõi, kế vị, truyền ngôi, lập thái tử, phế lập vua/hoàng hậu.
- POLITICAL_ADMINISTRATIVE: trọng tâm claim là chính sách, luật lệ, chiếu lệnh, quan chế, khoa cử, thuế khóa, bổ nhiệm quan lại, thiết chế nhà nước.
- MILITARY_DIPLOMACY: trọng tâm claim là chiến tranh, nổi loạn, đánh dẹp, phòng thủ, bang giao, triều cống, sứ thần, quan hệ với nước khác.
- NATURAL_OMEN_DISASTER: trọng tâm claim là thiên tai, thiên tượng, điềm lạ, nhật thực, nguyệt thực, sao Chổi, hạn hán, lũ lụt, động đất, dịch bệnh, mất mùa, giá gạo.

Nếu một claim có thể thuộc nhiều loại, chọn loại phản ánh trọng tâm chính cần kiểm chứng. Ví dụ: "Năm 1248, triều Trần đặt chức Hà đê sứ" có yếu tố thời gian nhưng trọng tâm là chức quan -> chọn POLITICAL_ADMINISTRATIVE.


=== CÁC LỖI TUYỆT ĐỐI TRÁNH ===
- Claim sao chép gần nguyên văn nguồn.
- Claim quá mơ hồ, thiếu chủ thể.
- Claim không được tự ý thêm tên riêng của chủ thể nếu source context không cung cấp rõ.
- Claim chứa nhiều fact độc lập trong một câu.
- Claim dùng đại từ mơ hồ ("ông ấy", "vua ấy", "người đó").
- Claim chủ quan hoặc mang tính khen chê không quy thuộc rõ nguồn.
- Claim REFUTED chỉ tạo bằng cách thêm "không".
- Claim NEI quá xa nguồn.
- Claim dựa vào kiến thức ngoài nguồn thay vì dựa vào đoạn nguồn được cung cấp.

=== ĐỊNH DẠNG OUTPUT ===
Chỉ trả về JSON hợp lệ theo đúng schema sau, không thêm markdown, không thêm giải thích ngoài JSON:
{
  "claims": [
    {
      "claim": "...",
      "label": "SUPPORTED | REFUTED | NOT_ENOUGH_EVIDENCE",
      "claim_type": "TIME_EVENT | PERSON_ROLE | GENEALOGY_SUCCESSION | POLITICAL_ADMINISTRATIVE | MILITARY_DIPLOMACY | NATURAL_OMEN_DISASTER",
      "explanation": "..."
    }
  ]
}
Nếu không tạo được claim nào phù hợp, trả về: {"claims": []}
"""

def build_generation_prompt(raw_text):
    return (
        f"{GEMINI_GENERATION_SYSTEM_PROMPT}\n\n"
        f"=== ĐOẠN NGUỒN SỬ LIỆU (source context) ===\n{raw_text}\n\n"
        f"Hãy ưu tiên tạo đủ 3 claim nếu đoạn nguồn có đủ điểm tựa cho cả 3 nhãn. "
        f"Chỉ trả JSON hợp lệ, không markdown."
    )

print("Đã nạp prompt sinh claim.")

Đã nạp prompt sinh claim.


In [ ]:
def load_input_chunks(path=INPUT_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy file '{path}'.")

    data = load_json(path)
    if not isinstance(data, list):
        raise ValueError("input_chunks.json phải là list JSON.")

    chunks = []
    input_issues = []

    for index, chunk in enumerate(data):
        if not isinstance(chunk, dict):
            input_issues.append({"row_index": index, "reason": "not_object"})
            continue

        missing = [field for field in REQUIRED_CHUNK_FIELDS if field not in chunk]
        if missing:
            input_issues.append({
                "row_index": index,
                "chunk_id": chunk.get("chunk_id"),
                "reason": "missing_required_fields",
                "missing_fields": missing,
            })
            continue

        item = dict(chunk)
        item["source_chunk_index"] = len(chunks)
        chunks.append(item)

    if MAX_CHUNKS_FOR_TEST is not None:
        chunks = chunks[:MAX_CHUNKS_FOR_TEST]

    log(f"Đọc {len(data)} dòng input.")
    log(f"Số chunk hợp lệ dùng để sinh claim: {len(chunks)}")
    log(f"Số dòng input lỗi: {len(input_issues)}")

    return chunks, input_issues

print("Đã nạp hàm đọc input.")

Đã nạp hàm đọc input.


## call gemini

In [ ]:
def call_gemini(prompt, temperature=GEMINI_TEMPERATURE):
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=genai_types.GenerateContentConfig(
                    temperature=temperature,
                    response_mime_type="application/json",
                ),
            )
            return response.text, None

        except Exception as exc:
            last_error = str(exc)
            log(f"Gemini lỗi {attempt}/{MAX_RETRIES}: {last_error}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS * attempt)

    return None, last_error

def validate_claim(raw_claim):
    if not isinstance(raw_claim, dict):
        return None, "format_error"

    claim_text = raw_claim.get("claim")
    label = normalize_label(raw_claim.get("label"))
    claim_type = normalize_claim_type(raw_claim.get("claim_type"))
    explanation = raw_claim.get("explanation")

    if not isinstance(claim_text, str) or not claim_text.strip():
        return None, "format_error"
    if label is None:
        return None, "format_error"
    if claim_type is None:
        return None, "format_error"
    if not isinstance(explanation, str):
        explanation = ""

    return {
        "claim": claim_text.strip(),
        "label": label,
        "claim_type": claim_type,
        "explanation": explanation.strip(),
    }, None

def generate_for_chunk(chunk):
    response_text, api_error = call_gemini(build_generation_prompt(chunk["raw_text"]))

    if response_text is None:
        return {
            "chunk_id": chunk["chunk_id"],
            "pages": chunk["pages"],
            "raw_text": chunk["raw_text"],
            "claims": [],
        }, [{
            "source_chunk_index": chunk["source_chunk_index"],
            "chunk_id": chunk["chunk_id"],
            "reason": "api_error",
            "detail": api_error,
        }], None

    parsed = extract_json_from_text(response_text)
    if not isinstance(parsed, dict) or not isinstance(parsed.get("claims"), list):
        return {
            "chunk_id": chunk["chunk_id"],
            "pages": chunk["pages"],
            "raw_text": chunk["raw_text"],
            "claims": [],
        }, [{
            "source_chunk_index": chunk["source_chunk_index"],
            "chunk_id": chunk["chunk_id"],
            "reason": "invalid_json_schema",
            "raw_response": response_text,
        }], response_text

    claims = []
    issues = []
    seen_labels = set()

    for claim_index, raw_claim in enumerate(parsed["claims"]):
        normalized, reason = validate_claim(raw_claim)
        if reason is not None:
            issues.append({
                "source_chunk_index": chunk["source_chunk_index"],
                "chunk_id": chunk["chunk_id"],
                "claim_index": claim_index,
                "reason": reason,
                "raw_claim": raw_claim,
            })
            continue

        label = normalized["label"]
        if label in seen_labels:
            issues.append({
                "source_chunk_index": chunk["source_chunk_index"],
                "chunk_id": chunk["chunk_id"],
                "claim_index": claim_index,
                "reason": "duplicate_label_in_same_chunk",
                "label": label,
                "raw_claim": raw_claim,
            })
            continue

        seen_labels.add(label)
        normalized["evidence_ids"] = [chunk["chunk_id"]]
        claims.append(normalized)

    claims.sort(key=lambda item: LABEL_ORDER.index(item["label"]))

    return {
        "chunk_id": chunk["chunk_id"],
        "pages": chunk["pages"],
        "raw_text": chunk["raw_text"],
        "claims": claims,
    }, issues, response_text

print("Đã nạp hàm sinh claim.")

Đã nạp hàm sinh claim.


In [ ]:
def run_generation(chunks):
    results = []
    issues = []
    raw_responses = []

    log(f"Bắt đầu sinh claim cho {len(chunks)} chunk.")

    for index, chunk in enumerate(chunks, start=1):
        log(f"[{index}/{len(chunks)}] {chunk['chunk_id']}")
        result, chunk_issues, response_text = generate_for_chunk(chunk)

        result["source_chunk_index"] = chunk["source_chunk_index"]
        results.append(result)
        issues.extend(chunk_issues)

        raw_responses.append({
            "source_chunk_index": chunk["source_chunk_index"],
            "chunk_id": chunk["chunk_id"],
            "response_text": response_text,
        })

        time.sleep(REQUEST_SLEEP_SECONDS)

    return results, issues, raw_responses

def build_draft_dataset(generated_by_chunk):
    dataset = []
    counter = 0

    for chunk_record in generated_by_chunk:
        for claim_index, claim_obj in enumerate(chunk_record["claims"], start=1):
            counter += 1
            dataset.append({
                "claim_id": f"claim_{counter:06d}",
                "source_chunk_index": chunk_record["source_chunk_index"],
                "claim_index_in_chunk": claim_index,
                "chunk_id": chunk_record["chunk_id"],
                "pages": chunk_record["pages"],
                "raw_text": chunk_record["raw_text"],
                "claim": claim_obj["claim"],
                "label": claim_obj["label"],
                "claim_type": claim_obj["claim_type"],
                "evidence_ids": claim_obj.get("evidence_ids") or [chunk_record["chunk_id"]],
                "explanation": claim_obj.get("explanation", ""),
                "human_review": "pending",
            })

    return dataset

print("Đã nạp hàm build dataset nháp.")

Đã nạp hàm build dataset nháp.


In [ ]:
def compute_stats(input_chunks, generated_by_chunk, draft_dataset, format_issues, input_issues):
    chunks_with_claims = sum(1 for item in generated_by_chunk if item["claims"])
    raw_claim_count = sum(len(item["claims"]) for item in generated_by_chunk) + sum(
        1 for item in format_issues
        if item.get("reason") in {"format_error", "duplicate_label_in_same_chunk"}
    )

    return {
        "num_input_chunks": len(input_chunks),
        "num_chunks_with_claims": chunks_with_claims,
        "num_chunks_without_claims": len(input_chunks) - chunks_with_claims,
        "num_claims_in_draft_dataset": len(draft_dataset),
        "num_pass1_format_issues": len(format_issues),
        "num_input_issues": len(input_issues),
        "pass1_issue_reason_counts": dict(Counter(item["reason"] for item in format_issues)),
        "label_distribution": dict(Counter(item["label"] for item in draft_dataset)),
        "claim_type_distribution": dict(Counter(item["claim_type"] for item in draft_dataset)),
        "claims_per_chunk_distribution": dict(Counter(len(item["claims"]) for item in generated_by_chunk)),
    }

def print_stats(stats):
    log("=== THỐNG KÊ ===")
    log(f"Chunk đầu vào: {stats['num_input_chunks']}")
    log(f"Chunk có claim: {stats['num_chunks_with_claims']}")
    log(f"Chunk không có claim: {stats['num_chunks_without_claims']}")
    log(f"Claim trong draft dataset: {stats['num_claims_in_draft_dataset']}")
    log(f"Format/API issue: {stats['num_pass1_format_issues']}")
    log(f"Phân bố label: {stats['label_distribution']}")
    log(f"Phân bố claim_type: {stats['claim_type_distribution']}")

print("Đã nạp hàm thống kê.")

Đã nạp hàm thống kê.


## main

In [ ]:
def main():
    ensure_output_dir()

    chunks, input_issues = load_input_chunks(INPUT_PATH)
    generated_by_chunk, format_issues, raw_responses = run_generation(chunks)
    draft_dataset = build_draft_dataset(generated_by_chunk)

    stats = compute_stats(
        input_chunks=chunks,
        generated_by_chunk=generated_by_chunk,
        draft_dataset=draft_dataset,
        format_issues=format_issues,
        input_issues=input_issues,
    )

    save_json(GEMINI_RAW_OUTPUTS_PATH, generated_by_chunk)
    save_json(DRAFT_DATASET_PATH, draft_dataset)
    save_json(PASS1_FORMAT_ISSUES_PATH, format_issues + input_issues)
    save_json(DATASET_STATS_PATH, stats)

    print_stats(stats)
    log(f"Đã lưu {GEMINI_RAW_OUTPUTS_PATH}")
    log(f"Đã lưu {DRAFT_DATASET_PATH}")
    log(f"Đã lưu {PASS1_FORMAT_ISSUES_PATH}")
    log(f"Đã lưu {DATASET_STATS_PATH}")

    return {
        "generated_by_chunk": generated_by_chunk,
        "draft_dataset": draft_dataset,
        "format_issues": format_issues,
        "input_issues": input_issues,
        "stats": stats,
    }

result = main()

[06:20:07] Đọc 494 dòng input.
[06:20:07] Số chunk hợp lệ dùng để sinh claim: 494
[06:20:07] Số dòng input lỗi: 0
[06:20:07] Bắt đầu sinh claim cho 494 chunk.
[06:20:07] [1/494] 019f17e0-fc5a-7cf2-ba14-b20bc5b39825
[06:20:09] [2/494] 019f17e0-fc85-73fc-b591-3981c10946ae
[06:20:24] [3/494] 019f17e0-fd57-7f7e-8051-5da239cf14fd
[06:20:27] [4/494] 019f17e0-fefe-74be-b3cc-0e4cab13c8bd
[06:20:29] [5/494] 019f17e1-0a34-793b-b170-d28a55c14360
[06:20:32] [6/494] 019f17e1-0a4b-787e-b48c-ea8c73fe0d10
[06:20:35] [7/494] 019f17e1-16f9-7209-b68a-a78071add7d2
[06:20:37] [8/494] 019f17e1-170e-735f-bddd-ccbf16280064
[06:20:40] [9/494] 019f17e1-18b2-7d22-b22d-83465c56e641
[06:20:43] [10/494] 019f17e1-2355-711c-ad0f-9862d5c52a0f
[06:20:45] [11/494] 019f17e1-2f77-7dc4-9640-0e8e2e686db3
[06:20:48] [12/494] 019f17e1-2fa7-7fac-9623-41435c8b6f90
[06:20:50] [13/494] 019f17e1-3b52-74b1-9868-5087c9550a67
[06:20:59] [14/494] 019f17e1-3b75-7b52-904b-d4c2cdb137d2
[06:21:01] [15/494] 019f17e1-467f-7fba-8417-49cebeb5